In [11]:
import pandas as pd
import numpy as np
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import accuracy_score, classification_report


df = pd.read_csv("data/processed_data.csv", parse_dates=["date"])
features = [
'diff_elo',
'diff_avg_goals_for', 
'diff_avg_goals_against',
'diff_avg_points', 
'diff_ranking',
'diff_fifa_points',
'ranking_local',
'ranking_away',
]

df = df[df["date"] < "2026-05-01"]

df_train = df[(df["date"] < "2023-06-01") ].copy()
X_train = df_train[features]
y_train = df_train["result"]
df_test = df[(df["date"] >= "2023-06-01") ].copy()
X_test = df_test[features]
y_test = df_test["result"]
y_test

modelo_gb = HistGradientBoostingClassifier(
    max_iter=100,  
    learning_rate=0.03,  # pasos cortos
    max_depth=3,  # limitamos la profundidad
    min_samples_leaf=70,  # Exigimos que al menos 30 partidos cumplan una regla para darla por válida
    random_state=45,
)

modelo_gb.fit(X_train, y_train)

predicciones = modelo_gb.predict(X_test)

# =================================================================
# PASO 4: Evaluar los resultados en consola
# =================================================================
# Calculamos el porcentaje total de aciertos
precision = accuracy_score(y_test, predicciones)

print(f"La precisión es de gradient tree boosting es: {precision * 100:.2f}%")
print("\nPor cada resultado (1=Local, 0=Empate, 2=Visitante):")
print(classification_report(y_test, predicciones))







La precisión es: 62.11%

Por cada resultado (1=Local, 0=Empate, 2=Visitante):
              precision    recall  f1-score   support

           0       0.67      0.01      0.01       275
           1       0.63      0.92      0.75       604
           2       0.60      0.59      0.59       343

    accuracy                           0.62      1222
   macro avg       0.63      0.51      0.45      1222
weighted avg       0.63      0.62      0.54      1222



In [ ]:
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from lightgbm import LGBMClassifier

models = {
    "Random Forest": RandomForestClassifier(
        n_estimators=150, 
        max_depth=5, 
        min_samples_leaf=50, 
        random_state=45
    ),
    "Gradient Boosting (Sklearn)": GradientBoostingClassifier(
        n_estimators=100, 
        learning_rate=0.03, 
        max_depth=3, 
        min_samples_leaf=50, 
        random_state=45
    ),
    "Regresión Logística": LogisticRegression(
        max_iter=1000, 
        multi_class='multinomial', 
        random_state=45
    ),
    "LightGBM": LGBMClassifier(
        n_estimators=100, 
        learning_rate=0.03, 
        max_depth=3, 
        min_child_samples=50, 
        random_state=45,
        models = {
            "Random Forest": RandomForestClassifier(
                n_estimators=150,
                max_depth=5,
                min_samples_leaf=50,
                random_state=45
            ),
            "Gradient Boosting (Sklearn)": GradientBoostingClassifier(
                n_estimators=100,
                learning_rate=0.03,
                max_depth=3,
                min_samples_leaf=50,
                random_state=45
            ),
            "Regresión Logística": LogisticRegression(
                max_iter=1000,
                multi_class='multinomial',
                random_state=45
            ),
            "LightGBM": LGBMClassifier(
                n_estimators=100,
                learning_rate=0.03,
                max_depth=3,
                min_child_samples=50,
                random_state=45,
                verbosity=-1
            ),
            "HistGradientBoostingClassifier": HistGradientBoostingClassifier(
                max_iter=100,
                learning_rate=0.03,
                max_depth=3,  # limitamos la profundidad
                min_samples_leaf=70,  # exigimos que al menos 70 partidos cumplan una regla
                random_state=45,
            )
        }

        # Diccionario para guardar los resultados finales
        resultados = []

        # Bucle para entrenar, predecir y evaluar cada modelo automáticamente
        for nombre, modelo in models.items():
            # Entrenar
            modelo.fit(X_train, y_train)
            # Predecir
            preds = modelo.predict(X_test)
            # Evaluar precisión general
            acc = accuracy_score(y_test, preds)
            
            # Guardar métricas en la lista
            resultados.append({
                "Modelo": nombre,
                "Accuracy Total": f"{acc * 100:.2f}%",
                "Métrica Numérica": acc
            })
            
            # Imprimir el reporte detallado en consola para revisar los empates (0)
            print(f"=== REPORTE DETALLADO: {nombre} ===")
            print(classification_report(y_test, preds, zero_division=0))
            print("-" * 50)

        # Crear un DataFrame para ver el ranking de modelos de forma limpia
        df_ranking_modelos = pd.DataFrame(resultados).sort_values(by="Métrica Numérica", ascending=False)
        df_ranking_modelos = df_ranking_modelos.drop(columns=["Métrica Numérica"])

        print("\n🏆 RANKING FINAL DE MEJORES MODELOS 🏆")
        print(df_ranking_modelos.to_string(index=False))
        max_depth=3,  # limitamos la profundidad
        min_samples_leaf=70,  # Exigimos que al menos 30 partidos cumplan una regla para darla por válida
        random_state=45,
    )

}

# Diccionario para guardar los resultados finales
resultados = []

# Bucle para entrenar, predecir y evaluar cada modelo automáticamente
for nombre, modelo in modelos.items():
    # Entrenar
    modelo.fit(X_train, y_train)
    # Predecir
    preds = modelo.predict(X_test)
    # Evaluar precisión general
    acc = accuracy_score(y_test, preds)
    
    # Guardar métricas en la lista
    resultados.append({
        "Modelo": nombre,
        "Accuracy Total": f"{acc * 100:.2f}%",
        "Métrica Numérica": acc
    })
    
    # Imprimir el reporte detallado en consola para revisar los empates (0)
    print(f"=== REPORTE DETALLADO: {nombre} ===")
    print(classification_report(y_test, preds, zero_division=0))
    print("-" * 50)

# Crear un DataFrame para ver el ranking de modelos de forma limpia
df_ranking_modelos = pd.DataFrame(resultados).sort_values(by="Métrica Numérica", ascending=False)
df_ranking_modelos = df_ranking_modelos.drop(columns=["Métrica Numérica"])

print("\n🏆 RANKING FINAL DE MEJORES MODELOS 🏆")
print(df_ranking_modelos.to_string(index=False))
